<a href="https://colab.research.google.com/github/Sumanth30-git/AI-Fake-Content-Detector/blob/main/Ai_Fake_Content_Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch gradio


In [ ]:
from transformers import pipeline

detector = pipeline("text-classification", model="jy46604790/Fake-News-Bert-Detect")

print("Model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: jy46604790/Fake-News-Bert-Detect
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully!


In [ ]:
def check_news(headline):
    result = detector(headline)[0]
    label = result['label']
    score = round(result['score'] * 100, 2)

    if label == "LABEL_1":
        verdict = "REAL NEWS"
    else:
        verdict = "FAKE NEWS"

    print(f"Headline : {headline}")
    print(f"Verdict  : {verdict}")
    print(f"Confidence: {score}%")
    print("-" * 50)

# Test it
check_news("Scientists confirm drinking coffee makes you immortal")
check_news("NASA launches new telescope to study distant galaxies")
check_news("Government announces new budget plan for 2025")

Headline : Scientists confirm drinking coffee makes you immortal
Verdict  : FAKE NEWS
Confidence: 99.89%
--------------------------------------------------
Headline : NASA launches new telescope to study distant galaxies
Verdict  : FAKE NEWS
Confidence: 96.84%
--------------------------------------------------
Headline : Government announces new budget plan for 2025
Verdict  : REAL NEWS
Confidence: 91.04%
--------------------------------------------------


In [ ]:
!pip install timm Pillow requests


In [ ]:
from transformers import pipeline
from PIL import Image
import requests
from io import BytesIO

# Load deepfake image detection model
img_detector = pipeline("image-classification",
                        model="dima806/deepfake_vs_real_image_detection")

print("Image model loaded!")


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Image model loaded!


In [ ]:
from google.colab import files

def check_uploaded_image():
    print("Select an image file from your computer...")
    uploaded = files.upload()

    for filename in uploaded.keys():
        image = Image.open(BytesIO(uploaded[filename])).convert("RGB")

        results = img_detector(image)
        top = results[0]
        label = top['label']
        score = round(top['score'] * 100, 2)

        if label == "real":
            verdict = "✅ REAL IMAGE"
        else:
            verdict = "🚨 DEEPFAKE IMAGE"

        print(f"\nFile       : {filename}")
        print(f"Verdict    : {verdict}")
        print(f"Confidence : {score}%")
        print("-" * 50)

check_uploaded_image()

Select an image file from your computer...


In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
from transformers import pipeline
from PIL import Image

# Load both models
print("Loading models...")
news_detector = pipeline("text-classification", model="jy46604790/Fake-News-Bert-Detect")
img_detector = pipeline("image-classification", model="dima806/deepfake_vs_real_image_detection")
print("Both models ready!")

# Fake news function
def detect_fake_news(headline):
    if not headline.strip():
        return "Please enter a headline."
    result = news_detector(headline)[0]
    label = result['label']
    score = round(result['score'] * 100, 2)
    verdict = "🚨 FAKE NEWS" if label == "LABEL_0" else "✅ REAL NEWS"
    return f"{verdict}\nConfidence: {score}%"

def detect_deepfake(image):
    if image is None:
        return "Please upload an image."
    results = img_detector(image)
    top = results[0]
    label = top['label']  # 'Real' or 'Fake'
    score = round(top['score'] * 100, 2)

    if label == "Real":  # capital R
        verdict = "✅ REAL IMAGE"
    else:
        verdict = "🚨 DEEPFAKE IMAGE"

    return f"{verdict}\nConfidence: {score}%"

# Build UI
with gr.Blocks(title="AI Fake Content Detector") as app:
    gr.Markdown("# 🤖 AI-Based Fake Content Detection System")
    gr.Markdown("Detect deepfake images and fake news using AI")

    with gr.Tab("📰 Fake News Detection"):
        gr.Markdown("### Enter a news headline to check")
        news_input = gr.Textbox(placeholder="Type a news headline here...")
        news_button = gr.Button("Analyze", variant="primary")
        news_output = gr.Textbox(label="Result")
        news_button.click(detect_fake_news, inputs=news_input, outputs=news_output)

    with gr.Tab("🖼️ Deepfake Image Detection"):
        gr.Markdown("### Upload a face image to check")
        img_input = gr.Image(type="pil")
        img_button = gr.Button("Analyze", variant="primary")
        img_output = gr.Textbox(label="Result")
        img_button.click(detect_deepfake, inputs=img_input, outputs=img_output)

app.launch(share=True)

Loading models...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: jy46604790/Fake-News-Bert-Detect
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Both models ready!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9f56590797afdb83a7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset
import pandas as pd

# Load fake news dataset from HuggingFace
print("Downloading dataset...")
dataset = load_dataset("GonzaloA/fake_news", split="test")
df = pd.DataFrame(dataset)
print(f"Dataset loaded: {len(df)} samples")
print(df['label'].value_counts())
print(df.head(3))

README.md:   0%|          | 0.00/6.73k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


dataset_infos.json:   0%|          | 0.00/1.03k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/38.8M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/13.0M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/13.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/24353 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8117 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8117 [00:00<?, ? examples/s]

Dataset loaded: 8117 samples
label
1    4335
0    3782
Name: count, dtype: int64
   Unnamed: 0                                              title  \
0           0  FORMER U.S. ATTORNEY: “It’s VERY Clear Intel C...   
1           1   Rick Santorum Advises President To Quit Tweet...   
2           2   Stephen Colbert Explains The Horrifying Futur...   

                                                text  label  
0  JOE DIGENOVA has been around D.C for decades a...      0  
1  In a recent interview with CNN, former senator...      0  
2  With only three clowns left in GOP clown car, ...      0  


In [ ]:
import pandas as pd
from tqdm import tqdm

# Take 100 fake + 100 real samples
fake_samples = df[df['label'] == 0].sample(100, random_state=42)
real_samples = df[df['label'] == 1].sample(100, random_state=42)
test_df = pd.concat([fake_samples, real_samples]).reset_index(drop=True)

print(f"Testing on {len(test_df)} samples (100 fake + 100 real)")
print("Running... this takes 5-10 minutes")

# Run batch testing
results = []
for i, row in tqdm(test_df.iterrows(), total=len(test_df)):
    try:
        headline = str(row['title'])[:512]  # limit length
        pred = news_detector(headline)[0]
        predicted = 0 if pred['label'] == 'LABEL_0' else 1
        results.append({
            'title': headline,
            'actual': row['label'],
            'predicted': predicted,
            'confidence': round(pred['score'] * 100, 2),
            'correct': row['label'] == predicted
        })
    except:
        pass

# Calculate metrics
results_df = pd.DataFrame(results)
accuracy = round(results_df['correct'].mean() * 100, 2)
fake_acc = round(results_df[results_df['actual']==0]['correct'].mean() * 100, 2)
real_acc = round(results_df[results_df['actual']==1]['correct'].mean() * 100, 2)

print(f"\n--- RESULTS ---")
print(f"Total tested : {len(results_df)}")
print(f"Overall accuracy : {accuracy}%")
print(f"Fake detection accuracy : {fake_acc}%")
print(f"Real detection accuracy : {real_acc}%")

# Save to CSV
results_df.to_csv('fake_news_results.csv', index=False)
print("\nSaved to fake_news_results.csv")

Testing on 200 samples (100 fake + 100 real)
Running... this takes 5-10 minutes


100%|██████████| 200/200 [00:29<00:00,  6.89it/s]


--- RESULTS ---
Total tested : 200
Overall accuracy : 85.5%
Fake detection accuracy : 99.0%
Real detection accuracy : 72.0%

Saved to fake_news_results.csv


In [ ]:
from google.colab import files
from PIL import Image
from io import BytesIO
import pandas as pd

print("Select ALL 50 images at once (25 real + 25 fake)")
print("Hold Ctrl and click to select multiple files")
uploaded = files.upload()

print(f"\nUploaded {len(uploaded)} images")

Select ALL 50 images at once (25 real + 25 fake)
Hold Ctrl and click to select multiple files


Saving download (1).jfif to download (1) (1).jfif
Saving download (2).jfif to download (2) (1).jfif
Saving download (3).jfif to download (3) (1).jfif
Saving download (4).jfif to download (4) (1).jfif
Saving download (5).jfif to download (5) (1).jfif
Saving download (6).jfif to download (6) (1).jfif
Saving download (7).jfif to download (7) (1).jfif
Saving download (8).jfif to download (8) (1).jfif
Saving download (9).jfif to download (9) (1).jfif
Saving download (10).jfif to download (10) (1).jfif
Saving download (11).jfif to download (11) (1).jfif
Saving download (12).jfif to download (12) (1).jfif
Saving download (13).jfif to download (13) (1).jfif
Saving download (14).jfif to download (14) (1).jfif
Saving download (15).jfif to download (15) (1).jfif
Saving download (16).jfif to download (16) (1).jfif
Saving download (17).jfif to download (17) (1).jfif
Saving download (18).jfif to download (18) (1).jfif
Saving download (19).jfif to download (19) (1).jfif
Saving download (20).jfif to d

In [ ]:
results = []

for filename, content in uploaded.items():
    try:
        image = Image.open(BytesIO(content)).convert("RGB")
        pred = img_detector(image)[0]
        label = pred['label'].lower()
        confidence = round(pred['score'] * 100, 2)

        # Fixed filename detection
        if 'whatsapp' in filename.lower():
            actual = 'real'
        elif 'download' in filename.lower():
            actual = 'fake'
        else:
            print(f"Skipping {filename} - can't determine actual label")
            continue

        correct = actual == label

        results.append({
            'filename': filename,
            'actual': actual,
            'predicted': label,
            'confidence': confidence,
            'correct': correct
        })
    except Exception as e:
        print(f"Error on {filename}: {e}")

results_df = pd.DataFrame(results)
accuracy = round(results_df['correct'].mean() * 100, 2)
fake_acc = round(results_df[results_df['actual']=='fake']['correct'].mean() * 100, 2)
real_acc = round(results_df[results_df['actual']=='real']['correct'].mean() * 100, 2)

print(f"--- RESULTS ---")
print(f"Total tested     : {len(results_df)}")
print(f"Overall accuracy : {accuracy}%")
print(f"Fake accuracy    : {fake_acc}%")
print(f"Real accuracy    : {real_acc}%")

results_df.to_csv('image_results.csv', index=False)
print("Saved to image_results.csv")

--- RESULTS ---
Total tested     : 61
Overall accuracy : 70.49%
Fake accuracy    : 41.94%
Real accuracy    : 100.0%
Saved to image_results.csv


In [ ]:
# Check what filenames we have
for filename in uploaded.keys():
    print(filename)


download (1) (1).jfif
download (2) (1).jfif
download (3) (1).jfif
download (4) (1).jfif
download (5) (1).jfif
download (6) (1).jfif
download (7) (1).jfif
download (8) (1).jfif
download (9) (1).jfif
download (10) (1).jfif
download (11) (1).jfif
download (12) (1).jfif
download (13) (1).jfif
download (14) (1).jfif
download (15) (1).jfif
download (16) (1).jfif
download (17) (1).jfif
download (18) (1).jfif
download (19) (1).jfif
download (20) (1).jfif
download (21) (1).jfif
download (22) (1).jfif
download (23) (1).jfif
download (24) (1).jfif
download (25) (1).jfif
download (26) (1).jfif
download (27) (1).jfif
download (28) (1).jfif
download (29) (1).jfif
download (30) (1).jfif
download (31).jfif
WhatsApp Image 2026-05-27 at 2.53.32 PM - Copy.jpeg
WhatsApp Image 2026-05-27 at 2.53.32 PM.jpeg
WhatsApp Image 2026-05-27 at 2.55.04 PM.jpeg
WhatsApp Image 2026-05-27 at 2.56.44 PM.jpeg
WhatsApp Image 2026-05-27 at 2.59.14 PM.jpeg
WhatsApp Image 2026-05-27 at 2.59.17 PM.jpeg
WhatsApp Image 2026-05-

In [ ]:
# Check 3 samples - 1 fake, 1 real
fake_file = 'download (1) (1).jfif'
real_file = 'WhatsApp Image 2026-05-27 at 2.53.32 PM.jpeg'

fake_img = Image.open(BytesIO(uploaded[fake_file])).convert("RGB")
real_img = Image.open(BytesIO(uploaded[real_file])).convert("RGB")

print("FAKE image result:")
print(img_detector(fake_img))

print("\nREAL image result:")
print(img_detector(real_img))

FAKE image result:
[{'label': 'Real', 'score': 0.7430518865585327}, {'label': 'Fake', 'score': 0.2569481134414673}]

REAL image result:
[{'label': 'Real', 'score': 0.9417480826377869}, {'label': 'Fake', 'score': 0.058251939713954926}]


In [ ]:
results = []

for filename, content in uploaded.items():
    try:
        image = Image.open(BytesIO(content)).convert("RGB")
        pred = img_detector(image)[0]
        label = pred['label']  # 'Real' or 'Fake' - keep original case
        confidence = round(pred['score'] * 100, 2)

        if 'WhatsApp' in filename:
            actual = 'Real'
        elif 'download' in filename.lower():
            actual = 'Fake'
        else:
            print(f"Skipping: {filename}")
            continue

        correct = actual == label

        results.append({
            'filename': filename,
            'actual': actual,
            'predicted': label,
            'confidence': confidence,
            'correct': correct
        })
    except Exception as e:
        print(f"Error: {filename}: {e}")

results_df = pd.DataFrame(results)
accuracy = round(results_df['correct'].mean() * 100, 2)
fake_acc = round(results_df[results_df['actual']=='Fake']['correct'].mean() * 100, 2)
real_acc = round(results_df[results_df['actual']=='Real']['correct'].mean() * 100, 2)

print(f"--- RESULTS ---")
print(f"Total tested     : {len(results_df)}")
print(f"Overall accuracy : {accuracy}%")
print(f"Fake accuracy    : {fake_acc}%")
print(f"Real accuracy    : {real_acc}%")

--- RESULTS ---
Total tested     : 61
Overall accuracy : 70.49%
Fake accuracy    : 41.94%
Real accuracy    : 100.0%
